# Clase 9 — Auxiliares · el gimnasio

9 ejercicios cortos sobre *Construir Market y su API*: drills para ganar soltura con las primitivas y profundizaciones opcionales.

⏱️ 🟢 núcleo ~6 min · 🔵 si vamos bien +6 min · 🟣 bonus +10 min.

### El gimnasio

Drills cortos para automatizar las primitivas de Python con datos de mercado, más una profundización final. Mismo formato de siempre: escribe tu código, ejecuta la **✅ comprobación plegada** (`Shift+Enter`) y, si te atascas, abre **💡 Ver solución**. Ninguno debería llevarte más de un par de minutos. No hacen falta para seguir el curso — pero te hacen rápido.

**Dosis mínima** = todo lo marcado **🟢 núcleo**: el calentamiento entero y los dos primeros drills de cada bloque. Lo **🔵 si vamos bien** y lo **🟣 bonus**, para volver otro día.

---

## 🏋️ Gimnasio · Calentamiento — repaso exprés de L8

Un disparo y su factura, antes de poner el tiempo en marcha.

### C1. Disparo exprés

<sub>🟢 núcleo · ~1 min</sub>

Contra el primer libro, envía una market buy de 0.05 con el engine y guarda `n_fills`.

<sub>practicas: repaso: market order</sub>

In [ ]:
from exchange.market import Market
from exchange.matching import MatchingEngine
from exchange.orders import Order, OrderType
book = Market.sample().step()
n_fills = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert n_fills is not None, '⏸ n_fills sigue en None: completa el ejercicio antes de validar'
assert n_fills >= 1
print('ok ->', n_fills)

<details>
<summary>💡 Ver solución</summary>

```python
engine = MatchingEngine()
fills = engine.process(Order('BTCUSDT', 'buy', 0.05, order_type=OrderType.MARKET), book)
n_fills = len(fills)
```

</details>

### C2. Factura exprés

<sub>🟢 núcleo · ~1 min</sub>

Con `fills = [(100.0, 1.0), (101.0, 1.0)]`, calcula `eff`.

<sub>practicas: repaso: precio efectivo</sub>

In [ ]:
fills = [(100.0, 1.0), (101.0, 1.0)]
eff = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert eff is not None, '⏸ eff sigue en None: completa el ejercicio antes de validar'
assert eff == 100.5
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
eff = sum(p * s for p, s in fills) / sum(s for p, s in fills)
```

</details>

---

## 🏋️ Gimnasio · Bloque 1 — El loop y la cuenta

step, submit, equity: el latido del backtest, drill a drill.

### A1. Contar el día

<sub>🟢 núcleo · ~2 min</sub>

Recorre `Market.sample()` con un while y cuenta los pasos en `n`.

<sub>practicas: el while de step()</sub>

In [ ]:
from exchange.market import Market
m = Market.sample()
n = 0

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert n == 500, 'el dia tiene 500 snapshots'
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
n = 0
while m.step() is not None:
    n = n + 1
```

</details>

### A2. Apertura y cierre

<sub>🟢 núcleo · ~2 min</sub>

Recorre el día y guarda `first_mid` y `last_mid`.

<sub>practicas: primer y último mid</sub>

In [ ]:
from exchange.market import Market
m = Market.sample()
first_mid = None
last_mid = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert first_mid is not None and last_mid is not None
m2 = Market.sample(); b = m2.step()
assert abs(first_mid - b.mid) < 1e-9
print(f'ok  {first_mid} -> {last_mid}')

<details>
<summary>💡 Ver solución</summary>

```python
first_mid = None
last_mid = None
while m.step() is not None:
    if first_mid is None:
        first_mid = m.book.mid
    last_mid = m.book.mid
```

</details>

### A3. La orden en su momento

<sub>🔵 si vamos bien · ~2 min</sub>

Avanza 10 pasos y en el 10º envía una market buy de 0.05 con `m.submit(...)`. Guarda `fills`.

<sub>practicas: submit dentro del loop</sub>

In [ ]:
from exchange.market import Market
from exchange.orders import Order, OrderType
m = Market.sample()
fills = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert fills is not None, '⏸ fills sigue en None: completa el ejercicio antes de validar'
assert fills and sum(f.size for f in fills) > 0
print('ok ->', fills[0])

<details>
<summary>💡 Ver solución</summary>

```python
for _ in range(10):
    m.step()
fills = m.submit(Order('BTCUSDT', 'buy', 0.05, order_type=OrderType.MARKET))
```

</details>

### A4. La cuenta tras el fill

<sub>🔵 si vamos bien · ~2 min</sub>

Aplica los fills dados a un `PositionTracker` y guarda `eq = tracker.equity(100000.0)`.

<sub>practicas: PositionTracker en el loop</sub>

In [ ]:
from exchange.portfolio import PositionTracker
from exchange.trades import Fill
fills = [Fill(1, 'BTCUSDT', 'buy', 99990.0, 0.1)]
eq = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert eq is not None, '⏸ eq sigue en None: completa el ejercicio antes de validar'
assert abs(eq - 1.0) < 1e-6, 'pagaste 9999, vale 10000: +1'
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
tracker = PositionTracker()
for f in fills:
    tracker.apply_fill(f)
eq = tracker.equity(100000.0)
```

</details>

### A5. Sin posición no hay historia

<sub>🔵 si vamos bien · ~2 min</sub>

Simula 5 pasos SIN operar, apuntando `tracker.equity(m.book.mid)` en `curva`. Comprueba que es plana en 0.

<sub>practicas: la curva plana</sub>

In [ ]:
from exchange.market import Market
from exchange.portfolio import PositionTracker
m = Market.sample()
tracker = PositionTracker()
curva = []

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert len(curva) == 5
assert all(abs(x) < 1e-9 for x in curva), 'sin fills, el equity no se mueve'
print('ok — la parte plana de la curva')

<details>
<summary>💡 Ver solución</summary>

```python
curva = []
for _ in range(5):
    m.step()
    curva.append(tracker.equity(m.book.mid))
```

</details>

---

## 🏋️ Para terminar — profundización

Trocear la ejecución a lo largo del loop: la semilla del VWAP.

### A6. Schedule de participación

<sub>🟣 bonus · ~5 min</sub>

Reparte una compra total de 1.0 en 10 trozos iguales, uno cada 50 pasos. Guarda `executed` (debe sumar ~1.0).

<sub>practicas: repartir un objetivo</sub>

In [ ]:
from exchange import Market, Order, Side, OrderType, PositionTracker
executed = 0.0

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert abs(executed - 1.0) < 1e-6
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
from exchange import Market, Order, Side, OrderType, PositionTracker
m = Market.sample(); i = 0; executed = 0.0
while True:
    book = m.step()
    if book is None: break
    if i % 50 == 0 and executed < 1.0 - 1e-9:
        for f in m.submit(Order('BTCUSDT', Side.BUY, 0.1, order_type=OrderType.MARKET)):
            executed += f.size
    i += 1
```

</details>

---

## 🏋️ Transferencia — la misma idea, otro dominio

El bucle de simulación (leer estado → decidir → acumular → avanzar) no es del mercado: es como se simula CUALQUIER sistema en el tiempo. Aquí, el consumo eléctrico de una casa.

### T1. El contador de la luz (mismo loop, otro sistema)

<sub>🟣 bonus · ~5 min</sub>

Una casa consume, cada hora, `lecturas[i]` kWh. Recórrelas con un bucle acumulando el total en `consumido` y, en paralelo, apuntando el acumulado en `curva` (como la equity curve, pero de kWh). Es el mismo patrón `step → acumula → avanza`.

<sub>practicas: bucle paso a paso fuera del trading</sub>

In [ ]:
lecturas = [0.4, 0.6, 1.2, 0.9, 0.3, 0.5]
consumido = 0.0
curva = []

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert abs(consumido - 3.9) < 1e-9, 'la suma de todas las lecturas'
assert curva == [0.4, 1.0, 2.2, 3.1, 3.4, 3.9], 'la curva es el acumulado paso a paso'
print('ok — un loop de simulación es un loop de simulación, venda kWh o BTC')

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

Un solo `for x in lecturas:` que haga `consumido += x` y luego `curva.append(consumido)`. Es la misma forma que la equity curve del ejercicio A5.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
consumido = 0.0
curva = []
for x in lecturas:
    consumido += x
    curva.append(consumido)
```

</details>

## Fin de los auxiliares

Vuelve al cuaderno principal cuando quieras.